# Verify an AI answer against its sources

**GroundLens** checks what an AI system says against the documents it was given, and keeps a record of that check that anyone can verify later.

This notebook takes one situation, an assistant answering a question about an invoice, in **English, German, French, Spanish and Italian**, and walks from `pip install` to a signed evidence record. You will see:

1. a wrong number caught in every language, whatever the number format;
2. what the evidence looks like, verifier by verifier;
3. the same evidence under two different policies, with two different decisions;
4. the record, its hash, and why it is the same on any machine.

Runs in Google Colab. Nothing here sends your text anywhere: the engine runs locally and never opens a network connection.

## 1. Install

`pip install groundlens` installs the GroundLens engine (GLV, for GroundLens Verification): the numeric and rules verifiers, the policy engine, the signed records and the command line. No dependencies, no network.

The **base bundle** is an optional, explicit download (about 470 MB, once): the multilingual encoder that powers the lexical verifier. It is the only command in the package that touches the network, and the download is checked against a hash pinned in the engine.

In [ ]:
%pip install -q groundlens
!groundlens --version

In [ ]:
from groundlens import Bundle

bundle = Bundle.installed("base")
if bundle is None:
    try:
        bundle = Bundle.pull("base")          # ≈470 MB, a minute or two in Colab
    except Exception as e:                    # noqa: BLE001 - keep the notebook running without it
        print("base bundle not installed:", e)
        print("Continuing with the numeric and rules verifiers only.")
print(Bundle.status("base"))

## 2. One situation, five languages

Same invoice, same question, same mistake: the assistant reads **ten thousand** as **one thousand**. Notice how differently the five languages write that number. GroundLens reads each one with the right locale and compares the quantities exactly, in base units.

In [ ]:
CASES = {
    "en": dict(
        question="What is the invoice total and when is it due?",
        source="The total amount due is 10,000 dollars, payable within 30 days of receipt.",
        answer="The invoice total is 1,000 dollars, due within 30 days of receipt.",
    ),
    "de": dict(
        question="Wie hoch ist der Rechnungsbetrag und wann ist er fällig?",
        source="Der Gesamtbetrag beläuft sich auf 10.000 Euro, zahlbar innerhalb von 30 Tagen nach Erhalt.",
        answer="Der Rechnungsbetrag beträgt 1.000 Euro, zahlbar innerhalb von 30 Tagen nach Erhalt.",
    ),
    "fr": dict(
        question="Quel est le montant de la facture et quand est-il dû ?",
        source="Le montant total s'élève à 10 000 euros, payable sous 30 jours à compter de la réception.",
        answer="Le montant de la facture est de 1 000 euros, payable sous 30 jours à compter de la réception.",
    ),
    "es": dict(
        question="¿Cuál es el importe de la factura y cuándo vence?",
        source="El importe total asciende a 10.000 euros, pagaderos en un plazo de 30 días desde la recepción.",
        answer="El importe de la factura es de 1.000 euros, pagaderos en un plazo de 30 días desde la recepción.",
    ),
    "it": dict(
        question="Qual è l'importo della fattura e quando scade?",
        source="L'importo totale ammonta a 10.000 euro, pagabili entro 30 giorni dal ricevimento.",
        answer="L'importo della fattura è di 1.000 euro, pagabili entro 30 giorni dal ricevimento.",
    ),
}

In [ ]:
from groundlens import verify

records = {}
for locale, case in CASES.items():
    records[locale] = verify(
        case["answer"],
        [("invoice.pdf#p1", case["source"])],
        question=case["question"],
        locale=locale,
    )
    print(f"[{locale}] {records[locale].decision}")
    print(records[locale].report())
    print()

Five `FAIL`s, one reason each: the numeric verifier found the answer's amount nowhere in the source and tells you which source number it lost to. The `30` days are supported in every language, so they do not appear in the report; only what a reviewer needs to look at does.

## 3. The evidence, not the verdict

A record keeps everything each verifier said about each claim. A verifier never decides. Look at the Spanish case in full:

In [ ]:
r = records["es"]
print(f"{'verifier':<20} {'result':<13} {'score':>6}  {'nearest source text':<24} notes")
for e in r.evidence:
    print(f"{e.verifier_id:<20} {e.result:<13} {e.score:>6.2f}  {str(e.source_text):<24} {', '.join(e.notes)}")

If the base bundle is installed you also see `groundlens.lexical` rows: one per content word, with the source word it is anchored to and a support score. Words like *importe* or *pagaderos* score high because the source says the same thing; the score is a contextual similarity from a frozen multilingual encoder, so the same word used differently scores lower, and the note `exact_string_in_span` tells you when the word is there verbatim anyway.

The 3.x API, `proofread()`, gives the same information as **receipts**: the weakest anchors first, each next to the source word it lost to.

In [ ]:
from groundlens import proofread

marks = proofread(CASES["fr"]["answer"], [("facture", CASES["fr"]["source"])], locale="fr", question=CASES["fr"]["question"], k=3)
print("floor:", round(marks.floor, 3), "| encoder:", marks.encoder_id)
print(marks.report())
for w in marks.warnings:
    print("warning:", w)

## 4. A correct answer, and a paraphrase

Verification is not string matching. A paraphrase with the right numbers passes; the numbers are compared as quantities, so `1,2 km` in the answer and `1200 m` in the source agree, and so do `212 °F` and `100 °C`.

In [ ]:
ok = verify(
    "Die Leitung ist 1,2 km lang und der Siedepunkt liegt bei 212 °F; der Umsatz betrug 37,35 Milliarden Dollar.",
    [("bericht", "Länge der Leitung: 1200 m. Siedepunkt: 100 °C.\nUmsatz (in Millionen Dollar)\n2024   37.350")],
    locale="de",
)
print(ok.decision)
for e in ok.evidence:
    if e.verifier_id == "groundlens.numeric":
        print(f"  {e.result:<10} {e.rationale}")

## 5. Policies decide

Which verifiers are required, what thresholds apply, what happens to a claim nobody could check: that is the **policy**, a short YAML file you control. The engine has no opinion. The same evidence under two policies can produce two decisions, and both are correct.

`eu_ai_act_high_risk_v1` ships with the package. It maps its outcomes to Art. 15(1) (accuracy and robustness) and Art. 12(1) (record keeping) of Regulation (EU) 2024/1689.

In [ ]:
from groundlens import Policy

case = CASES["it"]
strict = verify(case["answer"], [("fattura", case["source"])], question=case["question"], locale="it", policy="eu_ai_act_high_risk_v1")
print("eu_ai_act_high_risk_v1 →", strict.decision)
for m in strict.regulatory_mapping:
    print("   ", m["framework"], m["article"], "·", m["control"])

lenient = Policy.from_yaml('''
id: demo_review_only
version: 1.0.0
description: A numeric mismatch sends the answer to a human instead of failing it.
verifiers:
  required: [groundlens.numeric]
decision:
  any_contradiction_from: []
  unresolved_claims: REVIEW
  tolerate_unresolved_kinds: [word]
''')
review = verify(case["answer"], [("fattura", case["source"])], question=case["question"], locale="it", policy=lenient)
print("demo_review_only       →", review.decision)
print("same evidence?", [e.result for e in strict.evidence] == [e.result for e in review.evidence])

## 6. The record

Every verification is sealed in a record: input hashes, the verifiers and model hashes that ran, the evidence, the policy and its hash, the decision, the regulatory mapping, an Ed25519 signature. `content_hash` covers everything that is a function of input, policy and bundle, so two runs of the same check give the same hash on any machine; the record id and timestamp stay outside it.

In [ ]:
again = verify(CASES["en"]["answer"], [("invoice.pdf#p1", CASES["en"]["source"])], question=CASES["en"]["question"], locale="en")
print("content_hash  ", records["en"].content_hash)
print("second run    ", again.content_hash, "(same)" if again.content_hash == records["en"].content_hash else "(different!)")
print("record ids    ", records["en"].record_id, again.record_id)
print("policy        ", records["en"].policy_id, records["en"].policy_hash[:23] + "…")
print("bundle        ", records["en"].bundle_hash)
records["en"].verify()   # recomputes every hash and the signature, offline; raises if anything was altered
print("signature verifies")

In [ ]:
# Append the five records to a log. Each one carries the hash of the previous one.
from pathlib import Path
from groundlens import Record

log = Path("records.jsonl")
log.unlink(missing_ok=True)
for locale, case in CASES.items():
    verify(case["answer"], [("invoice.pdf#p1", case["source"])], question=case["question"], locale=locale, log=log)
print(Record.verify_chain(Record.read_log(log)), "records, chain intact")
!groundlens record verify records.jsonl

## Where next

* The second notebook, **Evidence records for auditors**, takes a log like this one and produces the report, the chain verification and the tamper test an auditor would run.
* Write your own policy with `Policy.from_yaml()` and your own rules (`rules=`) for the checks your domain needs.
* Documentation and source: [github.com/groundlens-dev/groundlens](https://github.com/groundlens-dev/groundlens).